# Politicians Network — Data Collection

Builds a **popularity-ranked** network of modern politicians (relevant after 1980).


In [1]:
import requests
import time
import json
import re
import pandas as pd
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from tqdm import tqdm
import threading
import random
from bs4 import BeautifulSoup


## 1 — Shared session & constants

In [2]:
S = requests.Session()
S.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

BASE_WIKI  = "https://en.wikipedia.org/w/api.php"
SPARQL_URL = "https://query.wikidata.org/sparql"

SKIP_PREFIXES = ("Special:", "Main_Page", "Wikipedia:", "Help:", "File:", "Portal:")


def get_with_retry(url, params=None, session=S, max_retries=6):
    """GET with exponential backoff on 429 / 5xx."""
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=30)
            if r.status_code == 429:
                wait = 2 ** attempt + random.uniform(0.5, 1.5)
                print(f"  Rate-limited, waiting {wait:.1f}s…")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise RuntimeError(f"Failed after {max_retries} retries: {url}") from e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


## 2 — SPARQL executor & helpers

In [3]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "WikidataResearchBot/1.0 (kerem.ozemre@icloud.com)"})


def run_sparql(query, retries=7, base_sleep=2.0):
    for attempt in range(retries):
        try:
            r = SESSION.post(
                SPARQL_URL,
                data={"query": query},
                headers={"Accept": "application/json",
                         "User-Agent": "WikidataResearchBot/1.0"},
                timeout=120,
            )
            if r.status_code == 429:
                wait = base_sleep * (2 ** attempt)
                print(f"  Wikidata 429 — waiting {wait:.0f}s (attempt {attempt+1})")
                time.sleep(wait)
                continue
            if r.status_code in (500, 502, 503, 504):   # ← added 504
                wait = base_sleep * (2 ** attempt)
                print(f"  Wikidata {r.status_code} — waiting {wait:.0f}s (attempt {attempt+1})")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.exceptions.Timeout:
            wait = base_sleep * (2 ** attempt)
            print(f"  Wikidata timeout (attempt {attempt+1}/{retries}) — waiting {wait:.0f}s")
            time.sleep(wait)
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(f"SPARQL failed: {e}")
            time.sleep(base_sleep * (2 ** attempt))
    raise RuntimeError(f"SPARQL failed after {retries} retries")


def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def parse_wikidata_date(raw):
    """Extract YYYY-MM-DD from a Wikidata timestamp string."""
    if not raw:
        return None
    m = re.match(r"[+-]?(\d{4}-\d{2}-\d{2})", raw)
    return m.group(1) if m else None


def extract_qid(uri):
    return uri.split("/")[-1]


## 3 — Popularity-ranked seed fetcher

Queries Wikidata for politicians ordered by **sitelink count** — the number of Wikipedia
language editions that cover them. This is the best available proxy for global notability
without hitting the pageviews API.

Filters: human (`P31=Q5`), politician (`P106=Q82955`), born ≤ 1970,
alive or died ≥ 1980.  
Remove `wdt:P27 wd:Q30` to go global instead of US-only.


In [4]:
def make_seed_query(limit=500, offset=0):
    return f"""
    SELECT DISTINCT ?person (COUNT(?sitelink) AS ?links) WHERE {{
      ?person wdt:P31 wd:Q5 ;       # person
              wdt:P106 wd:Q82955 . # occupation: politician

      ?person p:P39 ?stmt .         # held position (P39) statement
      ?stmt ps:P39 ?office .        
      ?office wdt:P17 wd:Q30 .          # office belongs to the United States

      OPTIONAL {{ ?person wdt:P569 ?birth. }} # date of birth
      OPTIONAL {{ ?person wdt:P570 ?death. }} # date of death  

      FILTER(
        BOUND(?birth) && YEAR(?birth) <= 1990
        && ( !BOUND(?death) || YEAR(?death) >= 1980 )
      )

      ?sitelink schema:about ?person .
    }}
    GROUP BY ?person
    ORDER BY DESC(?links) # most linked = most cross-wiki-covered
    LIMIT {limit}
    OFFSET {offset}
    """


def get_popular_politician_ids(max_seeds=800, page_size=500):
    """
    Return QID URI list for the most cross-wiki-covered politicians,
    paginating until max_seeds is reached or results are exhausted.
    page_size kept at 500 — GROUP BY + ORDER BY queries time out above ~500.
    """
    ids    = []
    offset = 0

    while len(ids) < max_seeds:
        query    = make_seed_query(limit=page_size, offset=offset)
        data     = run_sparql(query)
        bindings = data["results"]["bindings"]
        if not bindings:
            break
        batch = [b["person"]["value"] for b in bindings]
        ids.extend(batch)
        print(f"  Collected {len(ids)} seeds…")
        offset += page_size
        time.sleep(2)   # GROUP BY queries are heavier — be polite

    return ids[:max_seeds]


## 4 — Label fetcher

Resolves QIDs → English Wikipedia article titles.
The link-walker needs these titles to look up pages.
Also used to refresh titles for neighbor nodes discovered in expansion passes.


In [5]:
def fetch_labels(qids, batch_size=400):
    """
    Fetch English Wikipedia titles for a list of QIDs.
    Returns {wikipedia_title: qid} for QIDs that have an enwiki article.
    """
    name_to_qid = {}

    for chunk in tqdm(list(chunked(qids, batch_size)), desc="Fetching labels"):
        values = " ".join(f"wd:{q}" for q in chunk)
        query  = f"""
        SELECT ?person ?article WHERE {{
          VALUES ?person {{ {values} }}
          ?article schema:about ?person ;
                   schema:isPartOf <https://en.wikipedia.org/> .
        }}
        """
        data = run_sparql(query)
        for b in data["results"]["bindings"]:
            qid   = b["person"]["value"].split("/")[-1]
            title = (b["article"]["value"]
                     .replace("https://en.wikipedia.org/wiki/", "")
                     .replace("_", " "))
            name_to_qid[title] = qid
        time.sleep(1)

    print(f"  {len(name_to_qid)} QIDs have an English Wikipedia article")
    return name_to_qid


## 5 — Wikipedia link-walker (degree computation)

In [6]:
# ── Rate-limit governor ────────────────────────────────────────────────────────
_WIKI_SEM      = threading.Semaphore(4)
_BACKOFF_LOCK  = threading.Lock()
_BACKOFF_UNTIL = 0.0

def _wait_for_backoff():
    with _BACKOFF_LOCK:
        target = _BACKOFF_UNTIL
    remaining = target - time.monotonic()
    if remaining > 0:
        time.sleep(remaining)

def _set_backoff(seconds):
    global _BACKOFF_UNTIL
    with _BACKOFF_LOCK:
        _BACKOFF_UNTIL = max(_BACKOFF_UNTIL, time.monotonic() + seconds)


# ── Fetch links from article body only (no navboxes/tables) ───────────────────
def get_page_links_body_only(title):
    """
    Fetch only links that appear in article body prose.
    Strips navboxes, delegation tables, infoboxes — these are the source
    of the CA-delegation cluster artifact (e.g. every CA rep links to every
    other CA rep via the 'California delegation' navbox on their page).
    """
    session = requests.Session()
    session.headers.update({"User-Agent": "PoliticiansNetwork/1.0 (kerem.ozemre@icloud.com)"})

    params = {
        "action":    "parse",
        "page":      title,
        "prop":      "text",
        "format":    "json",
        "redirects": 1,
    }

    for attempt in range(6):
        try:
            r = get_with_retry("https://en.wikipedia.org/w/api.php",
                               params=params, session=session)
            html = r.json().get("parse", {}).get("text", {}).get("*", "")
            break
        except Exception as e:
            if attempt == 5:
                return []
            time.sleep(2 ** attempt)

    soup = BeautifulSoup(html, "html.parser")

    # Remove navboxes, tables, infoboxes, succession boxes, references
    for tag in soup.find_all(class_=[
        "navbox", "navbox-inner", "navbox-subgroup", "navbox-list",
        "wikitable", "infobox", "succession-box",
        "mw-references-wrap", "reflist", "portal",
        "noprint", "side-box", "sistersitebox",
    ]):
        tag.decompose()

    # Remove all <table> tags (catches delegation tables not covered above)
    for tag in soup.find_all("table"):
        tag.decompose()

    # Remove reference/external sections
    for section_id in ["References", "External_links", "See_also", "Further_reading"]:
        tag = soup.find(id=section_id)
        if tag:
            # Remove everything from this heading onward
            for sibling in tag.find_all_next():
                sibling.decompose()
            tag.decompose()

    # Collect links only from <p> paragraph tags
    links = []
    for p in soup.find_all("p"):
        for a in p.find_all("a", href=True):
            href = a["href"]
            if href.startswith("/wiki/") and ":" not in href:
                links.append(href[6:].replace("_", " "))

    return links


# ── Thread-safe caches ─────────────────────────────────────────────────────────
_title_lock = threading.Lock()
_qid_lock   = threading.Lock()

def resolve_titles_to_qids_cached(titles, cache, batch_size=50):
    with _title_lock:
        uncached = [t for t in titles if t not in cache]
    for i in range(0, len(uncached), batch_size):
        batch  = uncached[i:i + batch_size]
        params = {
            "action": "query", "titles": "|".join(batch),
            "prop": "pageprops", "ppprop": "wikibase_item", "format": "json",
        }
        r     = get_with_retry(BASE_WIKI, params=params)
        pages = r.json()["query"]["pages"]
        norm  = {n["from"]: n["to"] for n in r.json()["query"].get("normalized", [])}
        result = {}
        for page in pages.values():
            qid    = page.get("pageprops", {}).get("wikibase_item")
            ptitle = page.get("title", "")
            result[ptitle] = qid
        for orig, norm_title in norm.items():
            if norm_title in result:
                result[orig] = result[norm_title]
        with _title_lock:
            cache.update(result)
        time.sleep(0.1)
    with _title_lock:
        return {t: cache.get(t) for t in titles}


def filter_politician_qids_cached(qids, cache, batch_size=200):
    with _qid_lock:
        unchecked = [q for q in qids if q not in cache["seen"]]
    for i in range(0, len(unchecked), batch_size):
        batch  = unchecked[i:i + batch_size]
        values = " ".join(f"wd:{q}" for q in batch)
        query  = f"""
        SELECT ?person WHERE {{
          VALUES ?person {{ {values} }}
          ?person wdt:P106 wd:Q82955 .

          # Must have held a US office
          ?person p:P39 ?stmt .
          ?stmt ps:P39 ?office .
          ?office wdt:P17 wd:Q30 .

          # Same date filter as seeds
          OPTIONAL {{ ?person wdt:P569 ?birth. }}
          OPTIONAL {{ ?person wdt:P570 ?death. }}
          FILTER(
            BOUND(?birth) && YEAR(?birth) <= 1990
            && ( !BOUND(?death) || YEAR(?death) >= 1980 )
          )
        }}
        """
        data  = run_sparql(query)
        found = [item["person"]["value"].split("/")[-1]
                 for item in data["results"]["bindings"]]
        with _qid_lock:
            cache["politicians"].update(found)
            cache["seen"].update(batch)
        time.sleep(0.5)
    with _qid_lock:
        return cache["politicians"]


# ── Serial walker with adaptive throttle ──────────────────────────────────────
def compute_politician_degrees(name_to_qid, title_qid_cache=None, politician_cache=None):
    """
    Walk Wikipedia pages for each politician in name_to_qid.
    Returns (degrees, neighbours) dicts keyed by QID.
    Caches are shared across calls so repeated QIDs are only looked up once.
    """
    if title_qid_cache is None:
        title_qid_cache = {}
    if politician_cache is None:
        politician_cache = {"seen": set(), "politicians": set()}

    degrees   = {}
    neighbors = {}
    delay     = 0.5   # slightly higher base delay since parse API is heavier than links API

    for idx, (title, qid) in enumerate(tqdm(name_to_qid.items(), desc="Walking pages"), 1):
        linked_titles = []
        for attempt in range(6):
            try:
                linked_titles = get_page_links_body_only(title)
                delay = max(0.3, delay * 0.97)
                break
            except Exception as e:
                if "429" in str(e):
                    delay = min(30, delay * 2)
                    print(f"  429 on '{title}' — slowing to {delay:.1f}s")
                    time.sleep(delay)
                elif attempt == 5:
                    print(f"  Giving up on '{title}': {e}")
                    break
                else:
                    time.sleep(2 ** attempt)

        if not linked_titles:
            degrees[qid]   = 0
            neighbors[qid] = []
        else:
            t2q            = resolve_titles_to_qids_cached(linked_titles, title_qid_cache)
            linked_qids    = [q for q in t2q.values() if q]
            pol_set        = filter_politician_qids_cached(linked_qids, politician_cache) if linked_qids else set()
            pol_qids       = [q for q in linked_qids if q in pol_set]
            degrees[qid]   = len(pol_qids)
            neighbors[qid] = pol_qids

        time.sleep(delay)

        if idx % 50 == 0:
            print(f"  [{idx}/{len(name_to_qid)}] delay={delay:.2f}s | "
                  f"title cache={len(title_qid_cache)} | "
                  f"qid cache={len(politician_cache['seen'])}")

    return degrees,neighbors


## 6 — Feature fetcher

In [7]:
def make_details_query(person_ids):
    values = " ".join(f"wd:{p.split('/')[-1]}" for p in person_ids)
    return f"""
    SELECT
      ?person ?personLabel ?positionLabel
      ?start ?end ?natLabel ?birth ?death
      ?partyLabel ?genderLabel ?educationLabel ?stateLabel ?article
    WHERE {{
      VALUES ?person {{ {values} }}

      ?person p:P39 ?statement .
      ?statement ps:P39 ?position .
      OPTIONAL {{ ?statement pq:P580 ?start. }}
      OPTIONAL {{ ?statement pq:P582 ?end. }}

      OPTIONAL {{ ?person wdt:P27 ?nat. }}
      OPTIONAL {{ ?person wdt:P569 ?birth. }}
      OPTIONAL {{ ?person wdt:P570 ?death. }}
      OPTIONAL {{ ?person wdt:P102 ?party. }}
      OPTIONAL {{ ?person wdt:P21 ?gender. }}
      OPTIONAL {{ ?person wdt:P69 ?education. }}
      OPTIONAL {{ ?position wdt:P131 ?state. }}
      OPTIONAL {{
        ?article schema:about ?person ;
                 schema:isPartOf <https://en.wikipedia.org/> .
      }}

      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """


def fetch_details(person_ids, batch_size=50):
    rows = []
    for i, chunk in enumerate(chunked(person_ids, batch_size), 1):
        print(f"  Details batch {i}/{-(-len(person_ids)//batch_size)} ({len(chunk)} people)")
        data = run_sparql(make_details_query(chunk))
        for item in data["results"]["bindings"]:
            rows.append({
                "wikidata":      extract_qid(item["person"]["value"]),
                "name":          item.get("personLabel",   {}).get("value"),
                "position":      item.get("positionLabel", {}).get("value"),
                "start":         parse_wikidata_date(item.get("start",  {}).get("value")),
                "end":           parse_wikidata_date(item.get("end",    {}).get("value")),
                "nationality":   item.get("natLabel",      {}).get("value"),
                "birth_date":    parse_wikidata_date(item.get("birth",  {}).get("value")),
                "death_date":    parse_wikidata_date(item.get("death",  {}).get("value")),
                "party":         item.get("partyLabel",    {}).get("value"),
                "gender":        item.get("genderLabel",   {}).get("value"),
                "education":     item.get("educationLabel",{}).get("value"),
                "state":         item.get("stateLabel",    {}).get("value"),
                "wikipedia_url": item.get("article",       {}).get("value"),
            })
        time.sleep(1.5)
    return rows


## 7 — Dedup & party helpers

In [8]:
POSITION_RANK = {
    "President of the United States": 1,
    "Vice President of the United States": 2,
    "Secretary of State": 3,
    "Secretary of Defense": 4,
    "Attorney General of the United States": 5,
    "Secretary of the Treasury": 6,
    "White House Chief of Staff": 7,
    "United States Senator": 8,
    "Speaker of the House of Representatives": 9,
    "House Majority Leader": 10,
    "Senate Majority Leader": 10,
    "United States Representative": 11,
    "Governor": 12,
    "Ambassador": 13,
    "Lieutenant Governor": 14,
    "State Senator": 15,
    "State Representative": 16,
    "Mayor": 17,
}

def position_priority(pos):
    if pos is None:
        return 999
    if pos in POSITION_RANK:
        return POSITION_RANK[pos]
    for key, rank in POSITION_RANK.items():
        if key.lower() in pos.lower():
            return rank
    return 998
def simplify_party(p):
    if pd.isna(p):
        return "Other"

    p_lower = p.lower()

    # Major parties
    if "democratic" in p_lower:
        return "Democrat"
    if "republican" in p_lower:
        return "Republican" 
    return "Other"

# Positions that are clearly not elected offices
ADVISOR_POSITIONS = {
    "Senior Advisor to the President of the United States",
    "White House Chief Strategist",
    "Assistant to the President",
    "Counselor to the President",
    "Senior Counselor",
    "White House Press Secretary",
    "White House Communications Director",
    "Director of the Office of Management and Budget",
}




def dedup_df(df):
    """
    Per unique wikidata QID:
      - position  → most prestigious title (lowest POSITION_RANK score)
      - start     → earliest start date across all positions
      - end       → latest end date across all positions
      - all other columns (party, gender, education, etc.) → from the most prestigious row
    """
    df = df.copy()
    df["position_rank"] = df["position"].apply(position_priority)

    # ── Most prestigious row per person (for title + all metadata) ─────────────
    best = (df.sort_values("position_rank")
              .drop_duplicates(subset=["wikidata"], keep="first")
              .set_index("wikidata"))

    # ── Earliest start date per person ────────────────────────────────────────
    earliest_start = (df[df["start"].notna()]
                      .sort_values("start")
                      .drop_duplicates(subset=["wikidata"], keep="first")
                      .set_index("wikidata")["start"])

    # ── Latest end date per person ─────────────────────────────────────────────
    # Treat NaN end as still in office — represented as None, not overwritten
    latest_end = (df[df["end"].notna()]
                  .sort_values("end", ascending=False)
                  .drop_duplicates(subset=["wikidata"], keep="first")
                  .set_index("wikidata")["end"])

    # ── Assemble ───────────────────────────────────────────────────────────────
    best["career_start"] = best.index.map(earliest_start)
    best["career_end"]   = best.index.map(latest_end)

    # If someone has no end date at all, career_end stays None (still in office)
    # If someone has no start date at all, career_start stays None

    result = best.drop(columns=["start", "end", "position_rank"]).reset_index()

    

    print(f"  {len(result)} unique politicians after dedup")
    print(f"  career_start filled: {result['career_start'].notna().sum()}")
    print(f"    career_end filled: {result['career_end'].notna().sum()} "
          f"({result['career_end'].isna().sum()} still in office or unknown)")
    return result

## 8 — End-to-end pipeline


In [9]:
def run_pipeline(max_seeds=800, expand_neighbors=False):
    # ── Phase 1: popularity-ranked seeds ──────────────────────────────────────
    print("=" * 60)
    print("Phase 1 — Fetching popularity-ranked seed IDs…")
    seed_uris = get_popular_politician_ids(max_seeds=max_seeds)
    seed_qids = {extract_qid(u) for u in seed_uris}
    print(f"  Unique seeds: {len(seed_qids)}")

    print("\nFetching Wikipedia titles for seeds…")
    seed_titles = fetch_labels(list(seed_qids))
    print(f"  Seeds with enwiki article: {len(seed_titles)}")

    # Shared caches — reused across both expansion passes
    title_qid_cache  = {}
    politician_cache = {"seen": set(), "politicians": set()}

    # ── Phase 2: link-walk seeds ───────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("Phase 2 — Link-walking seed pages…")
    degrees, neighbours = compute_politician_degrees(
        seed_titles, title_qid_cache, politician_cache
    )

    neighbor_qids_1 = {q for conns in neighbours.values() for q in conns} - seed_qids
    all_qids        = seed_qids | neighbor_qids_1
    print(f"  Seeds              : {len(seed_qids)}")
    print(f"  New from pass 1    : {len(neighbor_qids_1)}")
    print(f"  Total after pass 1 : {len(all_qids)}")

    # ── Phase 3: optional second expansion pass ────────────────────────────────
    if expand_neighbors and neighbor_qids_1:
        print("\n" + "=" * 60)
        print("Phase 3 — Link-walking neighbor pages…")
        neighbor_titles = fetch_labels(list(neighbor_qids_1))
        degrees_2, neighbours_2 = compute_politician_degrees(
            neighbor_titles, title_qid_cache, politician_cache
        )

        neighbor_qids_2 = {q for conns in neighbours_2.values() for q in conns} - all_qids
        all_qids       |= neighbor_qids_2
        print(f"  New from pass 2    : {len(neighbor_qids_2)}")
        print(f"  Total after pass 2 : {len(all_qids)}")

        degrees.update(degrees_2)
        neighbours.update(neighbours_2)

    # ── Link-walk the non-seed neighbors too ──────────────────────────────────
    # Seeds already have degrees/neighbours from the walk above.
    # Non-seed nodes were only *discovered* as neighbors — we haven't walked
    # their pages yet, so we do that now to get their connections too.
    non_seed_qids   = all_qids - seed_qids
    non_seed_titles = fetch_labels(list(non_seed_qids))
    print(f"\nLink-walking {len(non_seed_titles)} non-seed pages for their connections…")
    degrees_ns, neighbours_ns = compute_politician_degrees(
        non_seed_titles, title_qid_cache, politician_cache
    )
    # Only add — don't overwrite seeds that were already walked
    for qid, deg in degrees_ns.items():
        if qid not in degrees:
            degrees[qid]   = deg
            neighbours[qid] = neighbours_ns[qid]

    # ── Phase 4: fetch features for everyone ──────────────────────────────────
    print("\n" + "=" * 60)
    print(f"Phase 4 — Fetching features for all {len(all_qids)} politicians…")
    all_uris = [f"http://www.wikidata.org/entity/{q}" for q in all_qids]
    rows     = fetch_details(all_uris)

    df = pd.DataFrame(rows)
    df = dedup_df(df)
    df["connections"] = df["wikidata"].map(neighbours).apply(
        lambda x: x if isinstance(x, list) else []
    )
    df["is_seed"] = df["wikidata"].isin(seed_qids)

    # ── Save edges to JSON ─────────────────────────────────────────────────────
    import json
    node_set     = set(df["wikidata"])
    clean_edges  = {
        qid: [n for n in conns if n in node_set]
        for qid, conns in neighbours.items()
        if qid in node_set
    }
    print(f"Saved → politician_edges.json  "
          f"({len(clean_edges)} nodes, "
          f"{sum(len(v) for v in clean_edges.values())} total edge entries)")

    # ── Save node CSV ──────────────────────────────────────────────────────────
    df.to_csv("politicians_network_nodes.csv", index=False)
    print(f"Saved → politicians_network_nodes.csv  ({len(df)} rows)")

    n_seeds = df["is_seed"].sum()
    print(f"\n{'=' * 60}")
    print(f"Pipeline complete.")
    print(f"  Seed nodes     : {n_seeds}")
    print(f"  Expanded nodes : {len(df) - n_seeds}")
    print(f"  Total nodes    : {len(df)}")
    return df


df = run_pipeline(max_seeds=1000, expand_neighbors=True)


Phase 1 — Fetching popularity-ranked seed IDs…
  Wikidata 504 — waiting 2s (attempt 1)
  Wikidata 504 — waiting 4s (attempt 2)
  Wikidata 504 — waiting 8s (attempt 3)
  Wikidata 429 — waiting 16s (attempt 4)
  Wikidata 504 — waiting 32s (attempt 5)
  Collected 500 seeds…
  Collected 1000 seeds…
  Unique seeds: 1000

Fetching Wikipedia titles for seeds…


Fetching labels: 100%|██████████| 3/3 [00:18<00:00,  6.10s/it]


  1000 QIDs have an English Wikipedia article
  Seeds with enwiki article: 1000

Phase 2 — Link-walking seed pages…


Walking pages:   2%|▏         | 16/1000 [01:08<42:36,  2.60s/it]  

  Wikidata 502 — waiting 2s (attempt 1)


...

  [5550/5625] delay=0.30s | title cache=111065 | qid cache=76669


Walking pages: 100%|█████████▉| 5600/5625 [2:24:01<00:33,  1.35s/it]

  [5600/5625] delay=0.30s | title cache=111192 | qid cache=76752


Walking pages: 100%|██████████| 5625/5625 [2:24:35<00:00,  1.54s/it]



Phase 4 — Fetching features for all 6625 politicians…
  Details batch 1/133 (50 people)
  Details batch 2/133 (50 people)
  Details batch 3/133 (50 people)
  Details batch 4/133 (50 people)
  Details batch 5/133 (50 people)
  Details batch 6/133 (50 people)
  Details batch 7/133 (50 people)
  Details batch 8/133 (50 people)
  Details batch 9/133 (50 people)
  Details batch 10/133 (50 people)
  Details batch 11/133 (50 people)
  Details batch 12/133 (50 people)
  Details batch 13/133 (50 people)
  Details batch 14/133 (50 people)
  Details batch 15/133 (50 people)
  Details batch 16/133 (50 people)
  Details batch 17/133 (50 people)
  Details batch 18/133 (50 people)
  Details batch 19/133 (50 people)
  Details batch 20/133 (50 people)
  Details batch 21/133 (50 people)
  Details batch 22/133 (50 people)
  Details batch 23/133 (50 people)
  Details batch 24/133 (50 people)
  Details batch 25/133 (50 people)
  Details batch 26/133 (50 people)
  Details batch 27/133 (50 people)
  Details

## 9 — Save

In [10]:
if not df.empty:
    print(f"Shape: {df.shape}")
    print(df.head(20).to_string())
else:
    print("Nothing to save — DataFrame is empty.")


Shape: (6625, 15)
     wikidata                        name                                              position    nationality  birth_date  death_date                      party  gender                                         education             state                                          wikipedia_url career_start  career_end                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         connections  is_seed
0   Q27996060              Stephen Miller  Senior Advisor to the President of the United States  United States  1985-08-23        None           Republican Party    male  

In [11]:
df["party"] = df["party"].apply(simplify_party)

In [12]:
career_end_dt = pd.to_datetime(df["career_end"], errors="coerce")

# Keep if: career_end is NaT (still active / unknown) OR career ended >= 1980
mask = career_end_dt.isna() | (career_end_dt >= pd.Timestamp("1980-01-01"))

df = df[mask].reset_index(drop=True)

In [13]:
df.to_csv("politicians_network_nodes_filtered.csv", index=False)